# Flask, APIs, and Jinja — Friendly Notes 🧩

This is Flask explanation file 2. It explains API CRUD actions, JSON, Jinja decisions and loops, forms, redirects, and dynamic URLs in small, simple pieces.

## 1. What is an API?

An **API** is like a small waiter between programs. A browser, phone app, or another program asks an API for data; the API sends a reply.

Instead of returning a pretty HTML page, an API often returns **JSON** — a neat text format for data. Python dictionaries and lists look very similar to JSON.

### JSON example

```json
{
  "id": 1,
  "name": "Buy milk",
  "description": "Get milk after school"
}
```

- Curly braces `{}` hold one object (like a Python dictionary).
- Square brackets `[]` hold a list of objects.
- JSON uses double quotes around keys and text values.

## 2. CRUD: the four list actions

Imagine a to-do list:

| CRUD action | HTTP method | Simple meaning | Example endpoint |
| --- | --- | --- | --- |
| **C**reate | `POST` | Add a new task | `POST /items` |
| **R**ead | `GET` | Look at tasks | `GET /items` |
| **U**pdate | `PUT` | Change a task | `PUT /items/2` |
| **D**elete | `DELETE` | Remove a task | `DELETE /items/2` |

An endpoint is simply an API door: a URL plus its HTTP method.

## 3. Start a tiny in-memory to-do API

The list below is only for learning. It is erased whenever the server restarts. In a real project, tasks would live in a database.

In [ ]:
from flask import Flask, jsonify, request

app = Flask(__name__)

items = [
    {"id": 1, "name": "Learn Flask", "description": "Read the notes"},
    {"id": 2, "name": "Make tea", "description": "Boil water"},
]

@app.route('/')
def home():
    return 'Welcome to the to-do API!'

if __name__ == '__main__':
    app.run(debug=True)

`jsonify(...)` turns Python data into a JSON response. Flask also adds the correct JSON response header for us.

In [ ]:
@app.route('/items', methods=['GET'])
def get_items():
    return jsonify(items)

@app.route('/items/<int:item_id>', methods=['GET'])
def get_item(item_id):
    item = next((thing for thing in items if thing['id'] == item_id), None)
    if item is None:
        return jsonify(error='Item not found'), 404
    return jsonify(item)

### Reading one item, slowly

`<int:item_id>` is a **variable URL part**. If someone opens `/items/2`, Flask calls `get_item(2)`.

`next(..., None)` finds the first matching task. If there is no match, it gives `None`; then we return a JSON error and status code **404**, which means “not found.”

In [ ]:
@app.route('/items', methods=['POST'])
def create_item():
    data = request.get_json(silent=True)
    if not data or not data.get('name'):
        return jsonify(error='A JSON body with a name is required'), 400

    new_item = {
        'id': (items[-1]['id'] + 1) if items else 1,
        'name': data['name'],
        'description': data.get('description', '')
    }
    items.append(new_item)
    return jsonify(new_item), 201

## 4. POST: add a task

Send this JSON body to `POST /items`:

```json
{
  "name": "Do homework",
  "description": "Finish the Flask exercise"
}
```

`request.get_json(silent=True)` reads JSON from the request safely. A successful create returns **201 Created**. A bad or missing request body returns **400 Bad Request**.

In [ ]:
@app.route('/items/<int:item_id>', methods=['PUT'])
def update_item(item_id):
    item = next((thing for thing in items if thing['id'] == item_id), None)
    if item is None:
        return jsonify(error='Item not found'), 404

    data = request.get_json(silent=True)
    if not data:
        return jsonify(error='A JSON body is required'), 400

    item['name'] = data.get('name', item['name'])
    item['description'] = data.get('description', item['description'])
    return jsonify(item)

@app.route('/items/<int:item_id>', methods=['DELETE'])
def delete_item(item_id):
    item = next((thing for thing in items if thing['id'] == item_id), None)
    if item is None:
        return jsonify(error='Item not found'), 404

    items.remove(item)
    return jsonify(message='Item deleted')

## 5. PUT and DELETE

- **PUT** changes an item that already exists. `data.get('name', item['name'])` means “use the new name if one was sent; otherwise keep the old name.”
- **DELETE** removes the matching item.

Use a client such as Postman, Insomnia, curl, or a frontend app to test API methods other than GET. The browser address bar alone only makes GET requests.

## 6. Jinja: tiny Python-like tools inside HTML

Jinja lets an HTML template show data from Flask.

- `{{ value }}` prints a value.
- `{% ... %}` performs a template action, such as an `if` or `for`.
- `{# ... #}` is a Jinja comment. Visitors cannot see it in the page.

Every opening `if` or `for` needs a matching ending tag.

### Display a dictionary with a loop

```html
{# templates/result.html #}
<h1>Result</h1>
{% for key, value in result.items() %}
  <p><strong>{{ key }}:</strong> {{ value }}</p>
{% endfor %}
```

Flask route:

```python
return render_template('result.html', result={'score': 55, 'status': 'Passed'})
```

The loop repeats the paragraph once for every key/value pair.

### Make a decision with `if`

```html
<p>Your score: {{ score }}</p>
{% if score >= 50 %}
  <h2>You passed! 🎉</h2>
{% else %}
  <h2>Not yet — keep practising.</h2>
{% endif %}
```

Be careful: Jinja tags need the exact punctuation, for example `{% endif %}` — not `end if`.

## 7. Form → calculate → redirect

A common journey is:

```text
GET /marks   → show form
POST /marks  → read form values and calculate
redirect     → go to a result URL
GET /result/<score> → show result page
```

A browser sends HTML form fields as text, so convert marks with `float(...)` or `int(...)` before adding them.

In [ ]:
from flask import render_template, redirect, url_for

@app.route('/marks', methods=['GET', 'POST'])
def marks():
    if request.method == 'POST':
        scores = [float(request.form[field]) for field in ('science', 'maths', 'english', 'data_science')]
        average = sum(scores) / len(scores)
        return redirect(url_for('show_result', score=average))
    return render_template('marks_form.html')

@app.route('/result/<float:score>')
def show_result(score):
    return render_template('marks_result.html', score=score)

`url_for('show_result', score=average)` builds the correct URL from the **function name**, instead of making you type a fragile URL by hand. This is called building a dynamic URL.

Example `templates/marks_form.html`:

```html
<form method="post">
  <input name="science" type="number" step="any" required>
  <input name="maths" type="number" step="any" required>
  <input name="english" type="number" step="any" required>
  <input name="data_science" type="number" step="any" required>
  <button>Calculate</button>
</form>
```

## 8. Recap checklist

- Use `jsonify` to return JSON from an API.
- Use GET to read, POST to create, PUT to update, and DELETE to remove.
- Give helpful JSON errors with appropriate status codes, such as 400 or 404.
- Put changing values in Jinja with `{{ ... }}`.
- Close Jinja blocks with `{% endfor %}` and `{% endif %}`.
- Read HTML form data with `request.form`.
- Use `redirect(url_for(...))` after a successful form submission.

### Mini challenge

Add a `done` value (`true`/`false`) to every task. Then make a `PUT /items/<id>` request that marks a task as done.